# OpenUBA SDK: ML Workflow

This notebook demonstrates the complete ML workflow: features, hyperparameters, experiments, and pipelines.

**SDK functions covered:** `create_features`, `load_features`, `create_hyperparameters`, `load_hyperparameters`, `create_experiment`, `add_experiment_run`, `compare_experiment_runs`, `create_pipeline`, `run_pipeline`

In [ ]:
import openuba
print(f'OpenUBA SDK v{openuba.__version__}')

## 1. Feature Engineering

Create feature groups for behavior analytics.

In [ ]:
# Create a feature group with individual features
try:
    feature_group = openuba.create_features(
        feature_names=[
            'login_count_24h',
            'unique_ips_24h',
            'failed_login_ratio',
            'off_hours_activity',
            'new_device_flag',
        ],
        group_name='nb-login-behavior',
        description='Login behavior features for anomaly detection',
        entity='user',
    )
    group_id = feature_group.get('id')
    print(f'Feature group created: {feature_group}')
except Exception as e:
    print(f'create_features: {e}')
    group_id = None

## 2. Load Features by Name

In [ ]:
try:
    loaded_group = openuba.load_features('nb-login-behavior')
    print(f'Loaded feature group:')
    for k, v in loaded_group.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'load_features: {e}')

## 3. Hyperparameter Management

Create and load hyperparameter configurations.

In [ ]:
hparam_ids = []
configs = [
    ('nb-iforest-conservative', {'contamination': 0.01, 'n_estimators': 100, 'max_samples': 256}),
    ('nb-iforest-balanced', {'contamination': 0.05, 'n_estimators': 200, 'max_samples': 512}),
    ('nb-iforest-aggressive', {'contamination': 0.10, 'n_estimators': 300, 'max_samples': 'auto'}),
]

for name, params in configs:
    try:
        hp = openuba.create_hyperparameters(
            name=name,
            parameters=params,
            description=f'Config: {name}',
        )
        hparam_ids.append(hp.get('id'))
        print(f'Created: {name} -> {hp.get("id")}')
    except Exception as e:
        print(f'create_hyperparameters ({name}): {e}')

print(f'\nTotal hyperparameter sets: {len(hparam_ids)}')

## 4. Load Hyperparameters

In [ ]:
if hparam_ids:
    try:
        loaded_hp = openuba.load_hyperparameters(hparam_ids[0])
        print(f'Loaded hyperparameters:')
        for k, v in loaded_hp.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'load_hyperparameters: {e}')
else:
    print('Skipped: no hparam_ids')

## 5. Experiment Tracking

Create an experiment and add runs with metrics.

In [ ]:
experiment_id = None
try:
    experiment = openuba.create_experiment(
        name='nb-iforest-tuning',
        description='Hyperparameter tuning for login anomaly detection',
    )
    experiment_id = experiment.get('id')
    print(f'Experiment created: {experiment}')
except Exception as e:
    print(f'create_experiment: {e}')

## 6. Add Experiment Runs

In [ ]:
if experiment_id:
    run_configs = [
        ({'contamination': 0.01, 'n_estimators': 100}, {'precision': 0.92, 'recall': 0.78, 'f1': 0.84, 'auc_roc': 0.91}),
        ({'contamination': 0.05, 'n_estimators': 200}, {'precision': 0.87, 'recall': 0.89, 'f1': 0.88, 'auc_roc': 0.94}),
        ({'contamination': 0.10, 'n_estimators': 300}, {'precision': 0.79, 'recall': 0.95, 'f1': 0.86, 'auc_roc': 0.93}),
    ]

    for i, (params, metrics) in enumerate(run_configs):
        try:
            run = openuba.add_experiment_run(
                experiment_id=experiment_id,
                parameters=params,
                metrics=metrics,
            )
            print(f'Run {i+1}: params={params} -> F1={metrics["f1"]}, AUC={metrics["auc_roc"]}')
        except Exception as e:
            print(f'add_experiment_run {i+1}: {e}')
else:
    print('Skipped: no experiment_id')

## 7. Compare Experiment Runs

In [ ]:
if experiment_id:
    try:
        comparison = openuba.compare_experiment_runs(experiment_id)
        print(f'Experiment comparison:')
        if isinstance(comparison, list):
            for run in comparison:
                m = run.get('metrics', {})
                p = run.get('parameters', {})
                print(f'  contamination={p.get("contamination")}: F1={m.get("f1")}, AUC={m.get("auc_roc")}')
        else:
            print(f'  {comparison}')
    except Exception as e:
        print(f'compare_experiment_runs: {e}')
else:
    print('Skipped: no experiment_id')

## 8. Pipeline Creation

In [ ]:
pipeline_id = None
try:
    pipeline = openuba.create_pipeline(
        name='nb-uba-pipeline',
        description='End-to-end UBA pipeline: preprocess -> train -> infer',
        steps=[
            {
                'type': 'training',
                'model_id': 'preprocessor',
                'hardware_tier': 'cpu-small',
                'hyperparameters': {'normalize': True, 'fill_na': 'median'},
            },
            {
                'type': 'training',
                'model_id': 'anomaly-detector',
                'hardware_tier': 'cpu-large',
                'hyperparameters': {'contamination': 0.05, 'n_estimators': 200},
            },
            {
                'type': 'inference',
                'model_id': 'anomaly-detector',
                'hardware_tier': 'cpu-small',
            },
        ],
    )
    pipeline_id = pipeline.get('id')
    print(f'Pipeline created:')
    for k, v in pipeline.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'create_pipeline: {e}')

## 9. Run Pipeline

In [ ]:
if pipeline_id:
    try:
        result = openuba.run_pipeline(pipeline_id)
        print(f'Pipeline run result: {result}')
    except Exception as e:
        print(f'run_pipeline: {e}')
else:
    print('Skipped: no pipeline_id')

In [ ]:
print('\n=== ML WORKFLOW COMPLETE ===')
print('Functions tested: create_features, load_features,')
print('  create_hyperparameters, load_hyperparameters,')
print('  create_experiment, add_experiment_run, compare_experiment_runs,')
print('  create_pipeline, run_pipeline')
print('All ML workflow checks passed!')